In [1]:
# stats for each (flux dates, phenocam dates, frequency, etc)
# located ecoregion level 3

In [1]:
import os

script_dir = os.path.dirname(os.path.abspath('load_utils.py'))

In [2]:
script_dir

'/Users/alexfache/Documents/GitHub/LCSC/analysis/planet'

In [4]:
import sys

for path in ['../../../LCSC/analysis/utils']:
    sys.path.append(path)
print(sys.path)
from load_paths import DIR_PLANET

['/opt/miniconda3/envs/LCSC/lib/python312.zip', '/opt/miniconda3/envs/LCSC/lib/python3.12', '/opt/miniconda3/envs/LCSC/lib/python3.12/lib-dynload', '', '/opt/miniconda3/envs/LCSC/lib/python3.12/site-packages', '../utils', '../../...LCSC/analysis/utils', '../../../LCSC/analysis/utils']
Darwin: (Darwin Kernel Version 24.6.0: Wed Oct 15 21:12:15 PDT 2025; root:xnu-11417.140.69.703.14~1/RELEASE_ARM64_T6041)
alexfache@mac.lan
device: MAC
DIR_DROPBOX: /Users/alexfache/Library/CloudStorage/Dropbox
DIR_LOCAL:   /Users/alexfache/Documents/GitHub/LCSC


In [5]:
import geopandas as gpd
import pandas as pd

# Load Site Data CSVs


In [6]:
phenocam_df = pd.read_csv(DIR_PLANET / 'qgis/sites/phenocam_sites.csv')
phenocam_df.rename(columns={c: f'(phenocam){c}' for c in list(phenocam_df.columns) if c not in ['site_name']}, inplace=True)
# phenocam_df

In [7]:
flux_df = pd.read_csv(DIR_PLANET / 'qgis/sites/AmeriFlux-site-search-results-202605032116.csv')
flux_df.rename(columns={c: f'(flux){c}' for c in list(flux_df.columns) if c not in ['Site ID']}, inplace=True)
# flux_df

In [8]:
study_sites_df = pd.read_csv(DIR_PLANET / 'qgis/sites/study_sites.csv')
study_sites_df[['latitude', 'longitude']] = study_sites_df['site_marker'].str.split(',', expand=True)
study_sites_df['latitude'] = study_sites_df['latitude'].astype(float)
study_sites_df['longitude'] = study_sites_df['longitude'].astype(float)
# study_sites_df

In [9]:
data_gdf = gpd.GeoDataFrame(study_sites_df, geometry=gpd.points_from_xy(study_sites_df.latitude, study_sites_df.longitude))
data_gdf = pd.merge(data_gdf, flux_df, on=['Site ID'], how='left')
data_gdf = pd.merge(data_gdf, phenocam_df, on=['site_name'], how='left')
data_gdf.rename(columns={'site_name': '(phenocam)site_name'}, inplace=True)
data_gdf.drop(columns=['(flux)Latitude (degrees)', '(flux)Longitude (degrees)'], axis=1, inplace=True)

# data_gdf

In [10]:
import json


def split_lat_lon_string(js):
    data = json.loads(js)
    return list(zip(data['lat'], data['lng']))


data_gdf['site_polygon'] = data_gdf['site_polygon'].apply(split_lat_lon_string)

In [11]:
data_gdf.to_file(DIR_PLANET / 'sites.gpkg', driver='GPKG')

/opt/miniconda3/envs/LCSC/lib/python3.12/site-packages/pyogrio/geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [12]:
data_gdf

,Flux Site Name,Site ID,(phenocam)site_name,site_url,site_marker,site_polygon,latitude,longitude,geometry,(flux)Name,...,(flux)Site End,(flux)BASE variables available,(flux)FLUXNET variables available,(phenocam)site_url,(phenocam)latitude,(phenocam)longitude,(phenocam)elevation_m,(phenocam)active,(phenocam)date_first,(phenocam)date_last
0,Walnut Gulch Kendall Grasslands (WKG),US-Wkg,kendall,https://ameriflux.lbl.gov/sites/siteinfo/US-Wkg,"31.7365,-109.9419","[(31.7814910017996, -109.994712510874), (31.78...",31.7365,-109.9419,POINT (31.736 -109.942),Walnut Gulch Kendall Grasslands,...,NaN,"CO2, FC, G, H, H2O, LE, LW_IN, LW_OUT, NETRAD,...",NaN,https://phenocam.nau.edu/webcam/sites/kendall/,31.73652,-109.94185,1529.0,True,2012-07-06,2026-03-23
1,Willard Juniper Savanna (WJS),US-Wjs,junipersavannah,https://ameriflux.lbl.gov/sites/siteinfo/US-Wjs,"34.4255,-105.8615","[(34.4704910017996, -105.915952483422), (34.47...",34.4255,-105.8615,POINT (34.426 -105.862),Willard Juniper Savannah,...,NaN,"CO2, FC, H, H2O, LE, LW_IN, LW_OUT, NETRAD, P,...",NaN,https://phenocam.nau.edu/webcam/sites/junipers...,34.42540,-105.86150,1931.0,True,2020-10-27,2026-03-23
2,Walnut Gulch Lucky Hills Shrub (WHS),US-Whs,luckyhills,https://ameriflux.lbl.gov/sites/siteinfo/US-Whs,"31.7438,-110.0522","[(31.7887910017996, -110.105016673341), (31.78...",31.7438,-110.0522,POINT (31.744 -110.052),Walnut Gulch Lucky Hills Shrub,...,NaN,"CO2, FC, G, H, H2O, LE, LW_IN, LW_OUT, NETRAD,...",NaN,https://phenocam.nau.edu/webcam/sites/luckyhills/,31.74390,-110.05202,1366.0,True,2013-04-30,2026-03-23
3,Santa Rita Mesquite (SRM),US-SRM,srm,https://ameriflux.lbl.gov/sites/siteinfo/US-SRM,"31.8214,-110.8661","[(31.8663910017996, -110.918961014599), (31.86...",31.8214,-110.8661,POINT (31.821 -110.866),Santa Rita Mesquite,...,NaN,"CO2, FC, G, H, H2O, LE, LW_IN, LW_OUT, NETRAD,...",NaN,https://phenocam.nau.edu/webcam/sites/srm/,31.82140,-110.86610,1116.0,True,2018-07-20,2026-03-23
4,Santa Rita Grassland (SRG),US-SRG,srg,https://ameriflux.lbl.gov/sites/siteinfo/US-SRG,"31.7894,-110.8277","[(31.8343910017996, -110.880542708782), (31.83...",31.7894,-110.8277,POINT (31.789 -110.828),Santa Rita Grassland,...,NaN,"CO2, FC, G, H, H2O, LE, LW_IN, LW_OUT, NETRAD,...",NaN,https://phenocam.nau.edu/webcam/sites/srg/,31.78940,-110.82760,1281.0,True,2019-09-04,2026-03-23
5,Sevilleta Shrubland (SES),US-Ses,sevilletashrub,https://ameriflux.lbl.gov/sites/siteinfo/US-SES,"34.3349,-106.7442","[(34.3798910017996, -106.798593602328), (34.37...",34.3349,-106.7442,POINT (34.335 -106.744),Sevilleta shrubland,...,NaN,"CO2, FC, H, H2O, LE, LW_IN, LW_OUT, NETRAD, P,...",NaN,https://phenocam.nau.edu/webcam/sites/sevillet...,34.33496,-106.74447,1603.0,True,2014-10-29,2026-03-23
6,Sevilleta Grassland (SEG),US-Seg,sevilletagrass,https://ameriflux.lbl.gov/sites/siteinfo/US-SEG,"34.3623,-106.7019","[(34.4072910017996, -106.756311381849), (34.40...",34.3623,-106.7019,POINT (34.362 -106.702),Sevilleta grassland,...,NaN,"CO2, FC, H, H2O, LE, LW_IN, LW_OUT, NETRAD, P,...",NaN,https://phenocam.nau.edu/webcam/sites/sevillet...,34.36044,-106.70019,1600.0,True,2014-11-05,2026-03-23
7,Mountainair Pinyon-Juniper Woodland (MPJ),US-Mpj,usmpj,https://ameriflux.lbl.gov/sites/siteinfo/US-MPJ,"34.4384,-106.2377","[(34.4833910017996, -106.292160888603), (34.48...",34.4384,-106.2377,POINT (34.438 -106.238),Mountainair Pinyon-Juniper Woodland,...,NaN,"CO2, FC, H, H2O, LE, LW_IN, LW_OUT, NETRAD, P,...",NaN,https://phenocam.nau.edu/webcam/sites/usmpj/,34.43845,-106.25436,2126.0,True,2013-09-20,2026-03-23
8,Konza Prairie LTER (KNZ),US-Kon,konza,https://ameriflux.lbl.gov/sites/siteinfo/US-KON,"39.0824,-96.5603","[(39.1273910017996, -96.6181632602185), (39.12...",39.0824,-96.5603,POINT (39.082 -96.56),Konza Prairie LTER (KNZ),...,NaN,"CO2, FC, G, GPP, H, H2O, LE, LW_IN, LW_OUT, NE...",NaN,https://phenocam.nau.edu/webcam/sites/konza/,39.08240,-96.56030,443.0,False,2012-03-17,2019-12-18
9,Southern Great Plains (ARM),US-ARM,southerngreat

In [13]:
data_gdf.columns

Index(['Flux Site Name', 'Site ID', '(phenocam)site_name', 'site_url',
       'site_marker', 'site_polygon', 'latitude', 'longitude', 'geometry',
       '(flux)Name', '(flux)Principal Investigator', '(flux)Data Use Policy',
       '(flux)AmeriFlux BASE Data', '(flux)AmeriFlux FLUXNET Data',
       '(flux)Vegetation Abbreviation (IGBP)',
       '(flux)Vegetation Description (IGBP)',
       '(flux)Climate Class Abbreviation (Koeppen)',
       '(flux)Climate Class Description (Koeppen)',
       '(flux)Mean Average Precipitation (mm)',
       '(flux)Mean Average Temperature (degrees C)', '(flux)Country',
       '(flux)Elevation (m)', '(flux)Number of years of AmeriFlux BASE data',
       '(flux)AmeriFlux BASE Data Start', '(flux)AmeriFlux BASE Data End',
       '(flux)Years of AmeriFlux BASE Data', '(flux)AmeriFlux BASE DOI',
       '(flux)AmeriFlux FLUXNET Data Start',
       '(flux)AmeriFlux FLUXNET Data End',
       '(flux)Years of AmeriFlux FLUXNET Data', '(flux)AmeriFlux FLUXNET DOI',

# Visualization of Sites


In [14]:
from ipyleaflet import AwesomeIcon, Map, Marker, Polygon, ScaleControl, basemaps
from ipywidgets import HTML, Layout

site_icon = AwesomeIcon(name='tower-cell', marker_color='blue', icon_color='blue', spin=False)
phenocam_icon = AwesomeIcon(name='camera', marker_color='green', icon_color='green', spin=False)

center = (44, -103)
m = Map(center=center, zoom=5, basemap=basemaps.Esri.WorldImagery, layout=Layout(height='500px'))
m.add(ScaleControl(position='bottomleft'))


site_boundary_locations = []

for index, site in data_gdf.iterrows():
    # Build hover tooltip HTML
    flux_name = site.get('(flux)Name', 'N/A')
    site_id = site.get('Site ID', 'N/A')
    veg = site.get('(flux)Vegetation Abbreviation (IGBP)', 'N/A')
    base_start = site.get('(flux)AmeriFlux BASE Data Start', 'N/A')
    base_end = site.get('(flux)AmeriFlux BASE Data End', 'N/A')
    flux_start = site.get('(flux)AmeriFlux FLUXNET Data Start', 'N/A')
    flux_end = site.get('(flux)AmeriFlux FLUXNET Data End', 'N/A')
    pheno_name = site.get('(phenocam)site_name', 'N/A')
    pheno_start = site.get('(phenocam)date_first', 'N/A')
    pheno_end = site.get('(phenocam)date_last', 'N/A')

    tooltip_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; min-width: 180px;">
        <b style="font-size: 13px;">Flux: {flux_name} ({site_id})</b><hr style="margin: 4px 0;">
        <b>Vegetation:</b> {veg}<br>
        <b>Base data:</b> {base_start} - {base_end}<br>
        <b>Flux data:</b> {flux_start} - {flux_end}<br>
        <hr style="margin: 4px 0;">
        <b>PhenoCam:</b> {pheno_name}<br>
        <b>PhenoCam data:</b> {pheno_start} - {pheno_end}
    </div>
    """

    marker = Marker(
        icon=site_icon,
        location=(site['latitude'], site['longitude']),
        draggable=True,
        title=f"{flux_name} ({site_id})",
    )

    # Attach popup that opens on hover
    popup = HTML(value=tooltip_html)
    marker.popup = popup

    m.add(marker)
    site_boundary_locations.append(site['site_polygon'])

    m.add(Marker(icon=phenocam_icon, location=(site['(phenocam)latitude'], site['(phenocam)longitude']), draggable=True, title=f'{site.get("(phenocam)site_name", "N/A")}'))

# Add polygons
site_boundary_polygons = Polygon(locations=site_boundary_locations, color='blue', fill_color='blue', fill_opacity=0.1, weight=2)
m.add(site_boundary_polygons)

display(m)

m.save('sites.html', title='My Map')

Map(center=[44, -103], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

# Create Selected Sites and Info CSV


In [ ]:
def underscore_name(row):
    return '_'.join(row.split(' '))

In [23]:
selected_sites = data_gdf[['(flux)Name']]
selected_sites.rename(columns={'(flux)Name': 'name'}, inplace=True)

selected_sites['site'] = selected_sites['name'].apply(underscore_name)

/var/folders/v_/1p9kfw391j985gnp_mvlhp880000gn/T/ipykernel_32162/3007803465.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_sites.rename(columns={'(flux)Name': 'name'}, inplace=True)
/var/folders/v_/1p9kfw391j985gnp_mvlhp880000gn/T/ipykernel_32162/3007803465.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_sites['site'] = selected_sites['name'].apply(underscore_name)


In [24]:
selected_sites.head(10)

,name,site
0,Walnut Gulch Kendall Grasslands,Walnut_Gulch_Kendall_Grasslands
1,Willard Juniper Savannah,Willard_Juniper_Savannah
2,Walnut Gulch Lucky Hills Shrub,Walnut_Gulch_Lucky_Hills_Shrub
3,Santa Rita Mesquite,Santa_Rita_Mesquite
4,Santa Rita Grassland,Santa_Rita_Grassland
5,Sevilleta shrubland,Sevilleta_shrubland
6,Sevilleta grassland,Sevilleta_grassland
7,Mountainair Pinyon-Juniper Woodland,Mountainair_Pinyon-Juniper_Woodland
8,Konza Prairie LTER (KNZ),Konza_Prairie_LTER_(KNZ)
9,ARM Southern Great Plains site- Lamont,ARM_Southern_Great_Plains_site-_Lamont
